# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors (FAIR⁲) Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR⁲ dataset using the `mlcroissant` library.

### Dataset Source
The dataset is defined by a Croissant schema and accessible from the following URL:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed in the environment
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL provided
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, including their `@id`s, fields, and sample records. All dataset entities are referenced by their `@id` according to Croissant best practices.

In [ ]:
# List all available RecordSets
from pprint import pprint

record_sets = list(dataset.record_sets)
print(f"Found {len(record_sets)} record sets.")
for rs in record_sets:
    print(f"- RecordSet name: {rs.name}, @id: {rs['@id']}")

# Show the fields (columns) of each RecordSet
for rs in record_sets:
    print(f"\nFields for RecordSet '{rs.name}' (id={rs['@id']}):")
    for field in rs.fields:
        print(f"  - {field.name} (@id: {field['@id']}) [type: {field.data_type}]")

# Preview a few records for each RecordSet (by @id)
for rs in record_sets:
    print(f"\nSample records for RecordSet '{rs.name}' (@id: {rs['@id']}):")
    for i, rec in enumerate(dataset.records(record_set=rs['@id'])):
        pprint(rec)
        if i >= 2:  # Show up to 3 records
            break

## 3. Data Extraction
Load data from each RecordSet (by `@id`) into pandas DataFrames for analysis. You can use any RecordSet or field `@id` directly as shown.

In [ ]:
# Extract all record sets using their @id
record_set_ids = [rs['@id'] for rs in dataset.record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)

# Show available columns for each extracted DataFrame
for rs_id, df in dataframes.items():
    print(f"\nColumns for RecordSet {rs_id}:")
    print(df.columns.tolist())
    print(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply processing and EDA steps on a selected RecordSet. We select numeric/categorical fields by their `@id` for further processing.

**Note**: Replace `<selected_record_set_id>`, `<numeric_field_id>`, and `<group_field_id>` with appropriate `@id` values from the overview section above.

In [ ]:
# Example: Select the main patient-level RecordSet and fields (update these IDs as appropriate for this dataset)
# For example purposes, we will use placeholder @ids. Replace with actual @id values from the overview output above.

# Suppose from the overview above, we found these (Update the variables below using output from previous cells):
selected_record_set_id = list(dataframes.keys())[0]  # Pick first record set by id as an example
df = dataframes[selected_record_set_id]

# Identify a numeric field @id (e.g., Age)
numeric_field_id = None
for col in df.columns:
    if 'age' in col.lower() or df[col].dtype in [int, float]:
        numeric_field_id = col
        break
if numeric_field_id is None:
    # Fallback to the first numeric-looking column
    numeric_field_id = df.select_dtypes(include=['int', 'float']).columns[0]

print(f"Using numeric field: {numeric_field_id}")

# Optionally, select a grouping field (e.g., Sex, MSI_Status). Again, ensure this is an @id.
group_field_id = None
for col in df.columns:
    if 'sex' in col.lower() or 'gender' in col.lower() or 'msi' in col.lower():
        group_field_id = col
        break

# Remove outliers: filter for reasonable values
mean_val = df[numeric_field_id].mean()
std_val = df[numeric_field_id].std()
# Simple outlier filter (within 3 std)
filtered_df = df[(df[numeric_field_id] > mean_val - 3*std_val) & (df[numeric_field_id] < mean_val + 3*std_val)]

print(f"Filtered DataFrame rows: {len(filtered_df)} / {len(df)}")

# Normalize numeric field
filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - mean_val) / std_val

print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

# Grouping and aggregate by group field if available
if group_field_id and group_field_id in df.columns:
    grouped = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
    print(f"\nGrouped mean {numeric_field_id} by {group_field_id}:")
    print(grouped.head())
else:
    print("No suitable grouping field found for grouping.")

## 5. Visualization
Visualize the distribution of the numeric field, and group means if grouping was performed. The following plots assume the fields and groupings discovered above.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution plot for the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[numeric_field_id], kde=True)
plt.title(f'Distribution of {numeric_field_id}')
plt.xlabel(numeric_field_id)
plt.show()

# If grouping performed, show means by group
if group_field_id and group_field_id in df.columns:
    plt.figure(figsize=(8, 5))
    sns.barplot(data=grouped, x=group_field_id, y=numeric_field_id)
    plt.title(f'Mean {numeric_field_id} by {group_field_id}')
    plt.ylabel(f'Mean {numeric_field_id}')
    plt.show()

## 6. Conclusion
This notebook demonstrated step-by-step how to load, inspect, and explore the FAIR⁲ tabular dataset using the `mlcroissant` library. You can adapt and extend these steps for any Croissant-format dataset by referencing entities via their `@id`, ensuring reproducible, schema-driven data processing and analysis.